# Multi-asset backtesting: contract-aware portfolio reports

`BacktestPriceTarget` emits `[equity, pnl, position, cost]` per asset. `PortfolioReport` reduces aligned asset rows into one causal portfolio report while retaining per-leg turnover and trade counts.

This seeded example uses different contract multipliers and fixed fees, as a futures portfolio might. The operator remains asset-agnostic: replace the synthetic paths with aligned engine output from any venue.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from screamer import BacktestPriceTarget, PortfolioReport

rng = np.random.default_rng(20)
n = 240
t = np.arange(n)
common = np.cumsum(rng.normal(0.0, 0.25, n))
nearby = 70.0 + common + 0.35 * np.sin(t / 15.0)
deferred = 71.5 + 0.8 * common + 0.25 * np.cos(t / 20.0)

# Offset legs make net-position turnover an invalid activity measure.
signal_nearby = np.where(np.sin(t / 22.0) >= 0.0, 1.0, -1.0)
signal_deferred = -0.5 * signal_nearby
nearby_output = BacktestPriceTarget(spread=0.0004, fee=0.0001, multiplier=50.0, fee_per_contract=0.02)(signal_nearby, nearby)
deferred_output = BacktestPriceTarget(spread=0.0006, fee=0.0001, multiplier=100.0, fee_per_contract=0.03)(signal_deferred, deferred)

# Shape: (time, assets, [equity, pnl, position, cost]).
engine_rows = np.stack([nearby_output, deferred_output], axis=1)
report = PortfolioReport()(engine_rows)
assert engine_rows.shape == (n, 2, 4)
assert report.shape == (n, 6)
print(f"{n} observations, {engine_rows.shape[1]} assets")

## Portfolio reduction

The six report columns are `[drawdown, cum_cost, turnover, trades, max_drawdown, sharpe]`. Turnover sums absolute position changes per leg, and `trades` counts legs whose position changed. This keeps offsetting spread or hedge activity visible.

In [ ]:
position = engine_rows[:, :, 2]
dposition = np.diff(position, axis=0, prepend=np.zeros((1, position.shape[1])))
expected_turnover = np.cumsum(np.abs(dposition).sum(axis=1))
expected_trades = np.cumsum((dposition != 0.0).sum(axis=1))
np.testing.assert_allclose(report[:, 2], expected_turnover)
np.testing.assert_allclose(report[:, 3], expected_trades)
print(f"final turnover: {report[-1, 2]:.2f} contracts")
print(f"leg trades counted: {report[-1, 3]:.0f}")

In [ ]:
fig, axes = plt.subplots(3, 1, sharex=True, figsize=(10, 7))
axes[0].plot(t, nearby, label="nearby", color="tab:blue")
axes[0].plot(t, deferred, label="deferred", color="tab:orange")
axes[0].set_ylabel("price")
axes[0].legend(loc="upper left")
axes[1].step(t, position[:, 0], where="post", label="nearby position", color="tab:blue")
axes[1].step(t, position[:, 1], where="post", label="deferred position", color="tab:orange")
axes[1].axhline(0.0, color="0.5", lw=0.7)
axes[1].set_ylabel("contracts")
axes[1].legend(loc="upper left")
axes[2].plot(t, report[:, 0], label="drawdown", color="tab:red")
axes[2].plot(t, report[:, 2], label="turnover", color="tab:green")
axes[2].set_xlabel("observation")
axes[2].legend(loc="upper left")
fig.suptitle("PortfolioReport keeps per-leg activity visible")
fig.tight_layout()

## Batch and streaming use the same reducer

A live strategy can feed one `(assets, 4)` row at a time. The lazy path uses the same C++ state machine as the batch path, so the results must agree.

In [ ]:
stream_report = np.asarray(list(PortfolioReport()(iter(engine_rows))))
np.testing.assert_allclose(stream_report, report, equal_nan=True)
summary = {
    "final_equity": float(engine_rows[-1, :, 0].sum()),
    "cumulative_cost": float(report[-1, 1]),
    "max_drawdown": float(report[-1, 4]),
    "final_turnover": float(report[-1, 2]),
    "leg_trades": float(report[-1, 3]),
}
summary

This notebook does not fetch data or assume a commodity calendar. It demonstrates the generic contract: aligned per-asset engine rows go in, fixed-shape portfolio report columns come out, and batch and streaming execution remain equivalent.